In [10]:
import numpy as np

## Class for Gaussian Elimination (with option for pivoting)

In [ ]:
class GaussianElim:
    def __init__(self, pivot = False):
        self.pivot = pivot
    
    def solveMat(self, A, b):
            # Convertimos a float para evitar truncamiento en la división
            Ab = np.hstack([A.astype(float), b.reshape(-1, 1).astype(float)])
            n = len(b)
            
            for i in range(n):
                if self.pivot: # Si hay pivoteo 
                    
                    max_row = np.argmax(np.abs(Ab[i:n, i])) + i
                    if i != max_row:
                        Ab[[i, max_row]] = Ab[[max_row, i]]
                
                # Verificación de matriz singular o pivote cero
                if np.isclose(Ab[i, i], 0):
                    raise ValueError("Pivote cero encontrado. El sistema no tiene solución única.")
                
                # Eliminación hacia adelante
                for j in range(i + 1, n):
                    factor = Ab[j, i] / Ab[i, i]
                    Ab[j, i:] -= factor * Ab[i, i:]
            
            # Sustitución hacia atrás
            x = np.zeros(n)
            for i in range(n - 1, -1, -1):
                x[i] = (Ab[i, -1] - np.dot(Ab[i, i+1:n], x[i+1:])) / Ab[i, i]
                
            return x

## Class for LU and PLU decomposition for solving, inverse and determinant calculations. 

In [16]:
class LuPlu:
    def __init__(self, pivot=False):
        self.pivot = pivot
        self.num_swaps = 0  # Contador para el signo del determinante

    def decomp(self, A):
        n = A.shape[0]
        U = A.copy().astype(float)
        L = np.eye(n)
        P = np.eye(n)
        self.num_swaps = 0
        
        for i in range(n):
            if self.pivot: # Si hay pivoteo
                
                max_row = np.argmax(np.abs(U[i:n, i])) + i
                if i != max_row:
                    U[[i, max_row]] = U[[max_row, i]] # Swaps
                    P[[i, max_row]] = P[[max_row, i]]
                    # Los multiplicadores en L calculados previamente también deben intercambiarse
                    if i > 0:
                        L[[i, max_row], :i] = L[[max_row, i], :i]
                    self.num_swaps += 1
            
            if np.isclose(U[i, i], 0):
                raise ValueError("Pivote cero encontrado. Utilice pivoteo.")
            
            # Descomposición
            for j in range(i + 1, n):
                factor = U[j, i] / U[i, i]
                L[j, i] = factor
                U[j, i:] -= factor * U[i, i:]
                
        if self.pivot:
            return P, L, U
        return L, U
    
    def _forward_sub(self, L, b): # Método sugerido por IA
        n = len(b)
        y = np.zeros_like(b, dtype=float)
        for i in range(n):
            y[i] = (b[i] - np.dot(L[i, :i], y[:i])) / L[i, i]
        return y
        
    def _backward_sub(self, U, y): # Método sugerido por IA
        n = len(y)
        x = np.zeros_like(y, dtype=float)
        for i in range(n - 1, -1, -1):
            x[i] = (y[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i]
        return x

    def solve(self, A, b):
        if self.pivot:
            P, L, U = self.decomp(A)
            # Para PLU: PA = LU -> LUx = Pb. Primero se resuelve Ly = Pb
            Pb = np.dot(P, b)
            y = self._forward_sub(L, Pb)
        else:
            L, U = self.decomp(A)
            # Para LU: LUx = b. Primero va Ly = b
            y = self._forward_sub(L, b)
            
        # Ux = y
        x = self._backward_sub(U, y)
        return x
    
    def det(self, A):
        if self.pivot:
            P, L, U = self.decomp(A)
            # det(A) = det(U) * (-1)^intercambios
            det_U = np.prod(np.diag(U))
            det_P = (-1)**self.num_swaps
            return det_U * det_P 
        else:
            L, U = self.decomp(A)
            return np.prod(np.diag(U))
    
    def inv(self, A):
        n = A.shape[0]
        I = np.eye(n)
        inv_A = np.zeros((n, n))
        
        # Resolvemos el sistema para cada columna de la matriz identidad
        for i in range(n):
            inv_A[:, i] = self.solve(A, I[:, i])
        return inv_A

## Exercises and Verification with Numpy (AI Enchanced)

In [15]:
# Inicialización de objetos
gauss_el = GaussianElim()
gauss_el_p = GaussianElim(pivot=True)
lu = LuPlu()
plu = LuPlu(pivot=True)

# Matriz A y vector b
A = np.array([
    [1, 2, -3, 4, 5], 
    [-2, -5, 8, -8, -9], 
    [1, 2, -2, 7, 9], 
    [1, 1, 0, 6, 12], 
    [2, 4, -6, 8, 11]
])
b = np.array([-1, -2, 4, 6, -3])

print("--- RESOLVIENDO EJERCICIOS ---")

# Ex. 1: Resuelva el sistema Ax = b utilizando eliminación gaussiana sin pivoteo
ex_1 = gauss_el.solveMat(A, b)
print("\nEx 1. Solución Gauss sin pivoteo:\n", ex_1)

# Ex. 2: Resuelva el sistema Ax = b utilizando eliminación gaussiana con pivoteo
ex_2 = gauss_el_p.solveMat(A, b)
print("\nEx 2. Solución Gauss con pivoteo parcial:\n", ex_2)

# Ex. 3: Descomposición LU (A = LU)
L_lu, U_lu = lu.decomp(A)
print("\nEx 3. Descomposición LU (Matriz L):\n", np.round(L_lu, 2))
print("Ex 3. Descomposición LU (Matriz U):\n", np.round(U_lu, 2))

# Ex. 4: Descomposición PLU (PA = LU)
P_plu, L_plu, U_plu = plu.decomp(A)
print("\nEx 4. Descomposición PLU (Matriz P):\n", P_plu)
print("Ex 4. Descomposición PLU (Matriz L):\n", np.round(L_plu, 2))
print("Ex 4. Descomposición PLU (Matriz U):\n", np.round(U_plu, 2))

# Ex. 5: Utilice descomposición LU para resolver Ax = b
ex_5 = lu.solve(A, b)
print("\nEx 5. Solución usando LU:\n", ex_5)

# Ex. 6: Utilice descomposición PLU para resolver Ax = b
ex_6 = plu.solve(A, b)
print("\nEx 6. Solución usando PLU:\n", ex_6)

# Ex. 7: Calcule el determinante de A usando PLU
ex_7 = plu.det(A)
print("\nEx 7. Determinante usando PLU:\n", round(ex_7, 4))

# Ex. 8: Calcule la inversa de A usando PLU
ex_8 = plu.inv(A)
print("\nEx 8. Inversa usando PLU:\n", np.round(ex_8, 4))

# =======================================================
# MÉTODO DE VERIFICACIÓN CONTRA NUMPY
# =======================================================
print("\n--- VERIFICACIÓN CON NUMPY ---")
# 1. Solución de Ax = b
np_x = np.linalg.solve(A, b)
assert np.allclose(ex_1, np_x), "Fallo en Gauss sin pivote"
assert np.allclose(ex_2, np_x), "Fallo en Gauss con pivote"
assert np.allclose(ex_5, np_x), "Fallo en LU solve"
assert np.allclose(ex_6, np_x), "Fallo en PLU solve"
print("Todas las soluciones x son correctas e iguales a np.linalg.solve")

# 2. Verificación de descomposiciones
assert np.allclose(A, L_lu @ U_lu), "Fallo en reconstituir A = LU"
print("Descomposición LU reconstruye exitosamente la matriz A (A = LU)")

assert np.allclose(P_plu @ A, L_plu @ U_plu), "Fallo en reconstituir PA = LU"
print("Descomposición PLU reconstruye exitosamente la matriz (PA = LU)")

# 3. Determinante e Inversa
np_det = np.linalg.det(A)
assert np.isclose(ex_7, np_det), f"Fallo en determinante: esperado {np_det}, obtenido {ex_7}"
print("Cálculo del determinante correcto")

np_inv = np.linalg.inv(A)
assert np.allclose(ex_8, np_inv), "Fallo en el cálculo de la matriz inversa"
print("Cálculo de la matriz inversa correcto")

--- RESOLVIENDO EJERCICIOS ---

Ex 1. Solución Gauss sin pivoteo:
 [-3. 69. 33. -8. -1.]

Ex 2. Solución Gauss con pivoteo parcial:
 [-3. 69. 33. -8. -1.]

Ex 3. Descomposición LU (Matriz L):
 [[ 1.  0.  0.  0.  0.]
 [-2.  1.  0.  0.  0.]
 [ 1. -0.  1.  0.  0.]
 [ 1.  1.  1.  1.  0.]
 [ 2. -0.  0. -0.  1.]]
Ex 3. Descomposición LU (Matriz U):
 [[ 1.  2. -3.  4.  5.]
 [ 0. -1.  2.  0.  1.]
 [ 0.  0.  1.  3.  4.]
 [ 0.  0.  0. -1.  2.]
 [ 0.  0.  0.  0.  1.]]

Ex 4. Descomposición PLU (Matriz P):
 [[0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]]
Ex 4. Descomposición PLU (Matriz L):
 [[ 1.    0.    0.    0.    0.  ]
 [-0.5   1.    0.    0.    0.  ]
 [-0.5   0.33  1.    0.    0.  ]
 [-1.    0.67 -1.    1.    0.  ]
 [-0.5   0.33 -0.5   0.5   1.  ]]
Ex 4. Descomposición PLU (Matriz U):
 [[-2.   -5.    8.   -8.   -9.  ]
 [ 0.   -1.5   4.    2.    7.5 ]
 [ 0.    0.    0.67  2.33  2.  ]
 [ 0.    0.    0.    1.   -1.  ]
 [ 0.    0.    0.    0.   -0.5 ]]
